# Used Cars Price Prediction in UAE

In [1]:
# Import necessary libraries

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

## 1. Load Dataset

In [2]:
df = pd.read_csv("yallamotor_used_cars_raw.csv")
df.head()

,Title,Brand,Model,Year,KM,Location,Price
0,بي إم دبليو X2 2022 مستعم...,بي,إم دبليو X2,2022,"64,027 كم",دبي,"61,699 درهم"
1,بي إم دبليو اكس1 2024 مست...,بي,إم دبليو اكس1,2024,"42,432 كم",دبي,"134,999 درهم"
2,بي إم دبليو اكس6 2023 مست...,بي,إم دبليو اكس6,2023,"64,298 كم",دبي,"194,999 درهم"
3,بي إم دبليو M8 Competitio...,بي,إم دبليو M8 Competitio,2023,"11,396 كم",دبي,"339,999 درهم"
4,بي إم دبليو اكس1 2018 مست...,بي,إم دبليو اكس1,2018,"117,873 كم",دبي,"45,299 درهم"


## 2. Explore the Dataset

In [3]:
df.info()

print("\nMissing values:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Title     216 non-null    object
 1   Brand     216 non-null    object
 2   Model     216 non-null    object
 3   Year      216 non-null    int64 
 4   KM        216 non-null    object
 5   Location  216 non-null    object
 6   Price     216 non-null    object
dtypes: int64(1), object(6)
memory usage: 11.9+ KB

Missing values:
Title       0
Brand       0
Model       0
Year        0
KM          0
Location    0
Price       0
dtype: int64


In [4]:
df['Brand'].value_counts()

Brand
بي         36
فورد       36
هيونداي    36
مرسيدس     36
نيسان      36
تويوتا     36
Name: count, dtype: int64

In [5]:
cars_brand = {
    "بي" : "bmw",
    "تويوتا" : "toyota",
    "مرسيدس" : "mercedes",
    "نيسان" : "nissan",
    "هيونداي" : "hyundai",
    "فورد" : "ford"
}

df.Brand=[cars_brand[x]for x in df.Brand]

df.head()

,Title,Brand,Model,Year,KM,Location,Price
0,بي إم دبليو X2 2022 مستعم...,bmw,إم دبليو X2,2022,"64,027 كم",دبي,"61,699 درهم"
1,بي إم دبليو اكس1 2024 مست...,bmw,إم دبليو اكس1,2024,"42,432 كم",دبي,"134,999 درهم"
2,بي إم دبليو اكس6 2023 مست...,bmw,إم دبليو اكس6,2023,"64,298 كم",دبي,"194,999 درهم"
3,بي إم دبليو M8 Competitio...,bmw,إم دبليو M8 Competitio,2023,"11,396 كم",دبي,"339,999 درهم"
4,بي إم دبليو اكس1 2018 مست...,bmw,إم دبليو اكس1,2018,"117,873 كم",دبي,"45,299 درهم"


In [6]:
df['Brand'].value_counts()

Brand
bmw         36
ford        36
hyundai     36
mercedes    36
nissan      36
toyota      36
Name: count, dtype: int64

In [7]:
df['Model'].value_counts()

Model
بنز الفئة أي              21
إم دبليو اكس1             12
إسكتيرا                   11
باليسايد                   9
Corolla Cross              9
Territory                  9
إم دبليو X2                6
سانتافيه                   6
توسان                      6
إلنترا                     6
بنز GLB                    6
إكسبيديشن                  6
إم دبليو 2 سيريز كوبيه     6
لاند كروز برادو 20         6
كيكس                       6
فورتشنر                    6
لاند كروز                  6
باثفايندر                  4
إكس تريل                   4
إكسبلورر                   3
Bronco                     3
إم دبليو X7                3
إم دبليو اكس4              3
إم دبليو M8 Competitio     3
إم دبليو اكس6              3
رينجر                      3
Everest                    3
أف-150                     3
إيدج                       3
كونا                       3
رينجر رابتور 3-0t-do       3
كامري                      3
يارس                       3
راف4                       3
باترول  

In [8]:
df['Location'].value_counts()

Location
دبي    216
Name: count, dtype: int64

## 3. Data Preprocessing

In [9]:
df["KM"] = (
    df["KM"]
    .str.replace("كم", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["Price"] = (
    df["Price"]
    .str.replace("درهم", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["KM"] = pd.to_numeric(df["KM"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

In [10]:
print(df.dtypes)

print("\nMissing values after conversion:")
print(df.isnull().sum())

df.head()

Title       object
Brand       object
Model       object
Year         int64
KM           int64
Location    object
Price        int64
dtype: object

Missing values after conversion:
Title       0
Brand       0
Model       0
Year        0
KM          0
Location    0
Price       0
dtype: int64


,Title,Brand,Model,Year,KM,Location,Price
0,بي إم دبليو X2 2022 مستعم...,bmw,إم دبليو X2,2022,64027,دبي,61699
1,بي إم دبليو اكس1 2024 مست...,bmw,إم دبليو اكس1,2024,42432,دبي,134999
2,بي إم دبليو اكس6 2023 مست...,bmw,إم دبليو اكس6,2023,64298,دبي,194999
3,بي إم دبليو M8 Competitio...,bmw,إم دبليو M8 Competitio,2023,11396,دبي,339999
4,بي إم دبليو اكس1 2018 مست...,bmw,إم دبليو اكس1,2018,117873,دبي,45299


## 4. Feature Selection

In [11]:
X = df[["Brand", "Model", "Year", "KM", "Location"]]
y = df["Price"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

X shape: (216, 5)
y shape: (216,)


,Brand,Model,Year,KM,Location
0,bmw,إم دبليو X2,2022,64027,دبي
1,bmw,إم دبليو اكس1,2024,42432,دبي
2,bmw,إم دبليو اكس6,2023,64298,دبي
3,bmw,إم دبليو M8 Competitio,2023,11396,دبي
4,bmw,إم دبليو اكس1,2018,117873,دبي


## 5. Encode Categorical Features

In [12]:
categorical_features = ["Brand", "Model", "Location"]
numeric_features = ["Year", "KM"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

## 6. Train-Test Split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (172, 5)
X_test: (44, 5)


## 7. Build Linear Regression Model

In [14]:
model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())])

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


## 8. Model Evaluation

In [15]:
y_pred = model.predict(X_test)

MAE = mean_absolute_error(y_test, y_pred)
MSE = mean_squared_error(y_test, y_pred)
RMSE = np.sqrt(MSE)
R2 = r2_score(y_test, y_pred)

print("MAE:", MAE)
print("MSE:", MSE)
print("RMSE:", RMSE)
print("R2 Score:", R2)

MAE: 12264.917392440395
MSE: 452141209.0455687
RMSE: 21263.61232353451
R2 Score: 0.9173674230584818


## 9. Actual vs Predicted Prices

In [16]:
results = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})

results.head(10)

,Actual Price,Predicted Price
0,60299,43535.567163
1,60299,43535.567163
2,154999,160501.892975
3,98399,96273.529961
4,339999,339988.521663
5,349999,265054.591422
6,34299,28126.458931
7,27799,29445.526944
8,84999,72471.516979
9,134999,120905.273492


## 10. Save Cleaned Dataset

In [17]:
df.to_csv("yallamotor_used_cars_cleaned.csv", index=False, encoding="utf-8-sig")

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


## Overall Conclusion

In this project, used car data was collected from YallaMotor using Web Scraping.

The final dataset contained 216 data points with important features such as Brand, Model, Year, Mileage, Location, and Price.

The data was preprocessed by cleaning the Price and KM columns, converting numerical features to numeric data types, and checking for missing values.

A Linear Regression model was built using Scikit-learn. Categorical features such as Brand, Model, and Location were encoded using OneHotEncoder.

The model achieved an R² score of approximately 0.917, which means that the model explains about 91.7% of the variation in used car prices.

The Mean Absolute Error was approximately 12,265 AED, showing that the predicted prices were generally close to the actual prices.

Overall, the model performed well and demonstrated that features such as car brand, model, year, mileage, and location can be useful for predicting used car prices.